In [ ]:
import subprocess, sys

for pkg in ["openai", "sentence-transformers", "faiss-cpu", "tabulate", "tqdm", "scikit-learn"]:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", pkg], check=True)

In [ ]:
import os, sys

REPO_URL    = "https://github.com/vudinhminh08/NLP-project-master-study.git"
REPO_BRANCH = "master"
PROJECT_DIR = "/kaggle/working/absa-project"

if not os.path.exists(PROJECT_DIR):
    !git clone --branch {REPO_BRANCH} --depth=1 {REPO_URL} {PROJECT_DIR}
else:
    !cd {PROJECT_DIR} && git pull origin {REPO_BRANCH}

os.chdir(PROJECT_DIR)
for d in ["data", "outputs/results", "outputs/llm_cache", "outputs/eda"]:
    os.makedirs(d, exist_ok=True)

sys.path.insert(0, os.path.join(PROJECT_DIR, "code", "week3"))
sys.path.insert(0, os.path.join(PROJECT_DIR, "code", "week1"))

In [ ]:
import pandas as pd

if not os.path.exists("data/train_preprocessed.csv"):
    !git clone https://github.com/ds4v/absa-vlsp-2018.git /tmp/ds4v --depth=1
    !cp /tmp/ds4v/datasets/vlsp2018_hotel/train.csv data/
    !cp /tmp/ds4v/datasets/vlsp2018_hotel/dev.csv data/
    !cp /tmp/ds4v/datasets/vlsp2018_hotel/test.csv data/

train_df = pd.read_csv("data/train_preprocessed.csv")
test_df  = pd.read_csv("data/test_preprocessed.csv")

In [ ]:
from kaggle_secrets import UserSecretsClient
OPENAI_API_KEY = UserSecretsClient().get_secret("KEY_GPT_MINHVU")

api_keys = {"openai": OPENAI_API_KEY}

In [ ]:
from icl_predictor import run_icl_ablation

tier2_results = run_icl_ablation(
    test_df=test_df,
    train_df=train_df,
    providers=["openai"],
    k_values=[2, 4, 8],
    api_keys=api_keys,
    models={"openai": "gpt-4o-mini"},
    results_dir="outputs/results",
    max_samples=None,
)

In [ ]:
from rag_predictor import run_rag_ablation

tier3_results = run_rag_ablation(
    test_df=test_df,
    train_df=train_df,
    providers=["openai"],
    k_values=[2, 4, 8],
    api_keys=api_keys,
    models={"openai": "gpt-4o-mini"},
    results_dir="outputs/results",
    max_samples=None,
)

In [ ]:
from compare_results import generate_comparison_table
print(generate_comparison_table(results_dir="outputs/results", save_path="outputs/results/week3_comparison.md"))

In [ ]:
import json, matplotlib.pyplot as plt

k_values = [2, 4, 8]
provider = "openai"

icl_f1s, rag_f1s = [], []
for k in k_values:
    icl = json.load(open(f"outputs/results/tier2_{provider}_k{k}_metrics.json"))
    rag = json.load(open(f"outputs/results/tier3_{provider}_k{k}_metrics.json"))
    icl_f1s.append(icl["macro_combined_f1"])
    rag_f1s.append(rag["macro_combined_f1"])

PHOBERT_F1 = 0.5543
SVM_F1     = 0.3173

fig, ax = plt.subplots(figsize=(8, 5))
ax.plot(k_values, icl_f1s, "o-", color="steelblue",  label="ICL (random)", lw=2)
ax.plot(k_values, rag_f1s, "o-", color="darkorange", label="RAG (retrieval)", lw=2)
ax.axhline(y=PHOBERT_F1, color="green", ls="--", label=f"PhoBERT ({PHOBERT_F1:.4f})", alpha=0.7)
ax.axhline(y=SVM_F1,     color="gray",  ls=":",  label=f"SVM ({SVM_F1:.4f})", alpha=0.7)
ax.set(title="GPT-4o-mini — ICL vs RAG (k ablation)", xlabel="k (số examples)", ylabel="Combined F1")
ax.set_xticks(k_values)
ax.legend()
ax.grid(alpha=0.3)
plt.tight_layout()
plt.savefig("outputs/eda/week3_ablation.png", dpi=150, bbox_inches="tight")
plt.show()

In [ ]:
import shutil
shutil.make_archive("/kaggle/working/week3_results", "zip", "outputs/results")